# 05 — Inspect Drizzled Outputs

**Purpose:** Review the final drizzled mosaics for all 6 quasars. Check image quality,
pixel scale, astrometry, and signal-to-noise. Flag any quasars that need re-drizzling.

**Inputs:** DRZ files from `data/processed/<quasar>/drizzled/`

**Outputs:** Diagnostic plots; updates to `docs/quasars.md` via `/summarize-results`

**After running:** Use `/summarize-results` to generate the final results table.

## Imports

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import ZScaleInterval
from astropy.wcs import WCS

## Define paths

In [ ]:
PROCESSED_DIR = Path('../data/processed')
quasar_dirs = sorted([d for d in PROCESSED_DIR.iterdir() if d.is_dir()])
print(f'Quasars found: {[d.name for d in quasar_dirs]}')

## Plot drizzled mosaics

In [ ]:
zscale = ZScaleInterval()

for qdir in quasar_dirs:
    drz_files = sorted((qdir / 'drizzled').glob('*drz_sci.fits'))
    if not drz_files:
        print(f'{qdir.name}: no DRZ science files found')
        continue
    
    for drz in drz_files:
        with fits.open(drz) as hdul:
            sci = hdul['SCI'].data
            hdr = hdul['SCI'].header
            wcs = WCS(hdr)
        
        vmin, vmax = zscale.get_limits(sci)
        fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'projection': wcs})
        ax.imshow(sci, vmin=vmin, vmax=vmax, origin='lower', cmap='gray')
        ax.set_title(f'{qdir.name}\n{drz.name}', fontsize=9)
        ax.coords[0].set_axislabel('RA')
        ax.coords[1].set_axislabel('Dec')
        plt.tight_layout()
        plt.show()

## Output summary

In [ ]:
print(f'{"Quasar":<20} {"Shape":<16} {"Pixel Scale":<14} {"DRZ File"}')
print('-' * 70)

for qdir in quasar_dirs:
    drz_files = sorted((qdir / 'drizzled').glob('*drz_sci.fits'))
    for drz in drz_files:
        with fits.open(drz) as hdul:
            sci = hdul['SCI'].data
            hdr = hdul['SCI'].header
            scale = abs(hdr.get('CD1_1', hdr.get('CDELT1', 0))) * 3600
        print(f'{qdir.name:<20} {str(sci.shape):<16} {scale:.4f}"/pix      {drz.name}')

## Next steps

1. Review the mosaics above. If any quasar needs re-drizzling with different parameters,
   edit `config/wfc3_ir_drizzle_params.yaml` and re-run `04_drizzle.ipynb`.
2. Run `/summarize-results` to generate the final results table in `docs/quasars.md`.
3. Run `/end-session` before closing this chat window.